# Calidad de Datos (Data Quality Gate) con Great Expectations

En nuestro pipeline diseñado bajo la Arquitectura Medallion, los datos operacionales de los incidentes de la tabla PROBLEMA aterrizan en la capa Bronze en formato crudo sin tipar. Antes de consolidar y promover la información hacia la capa Silver, el pipeline corre de forma distribuida un conjunto de reglas de aserción de calidad utilizando **Great Expectations**.

Este proceso actúa como un Data Quality Gate, garantizando que la base de datos analítica no se contamine con anomalías de negocio y derivando automáticamente los registros corruptos hacia una Dead Letter Queue (DLQ) para su posterior remediación por parte del Data Engineer.

In [ ]:
# Configuración Inicial
!pip install great_expectations pandas --quiet

import pandas as pd
import great_expectations as gx

In [ ]:
# Datos reales provenientes del DML relacional (Parte 1 del TP)
datos_reales = [
    [1,  'Datos duplicados en tabla de clientes',                       '2022-03-15','Resuelto',  2, 3],
    [2,  'Valores nulos en campo ingreso_neto',                         '2022-07-22','Resuelto',  1, 1],
    [3,  'Latencia excesiva en pipeline de ventas',                     '2022-11-10','Resuelto',  2, 6],
    [4,  'Error de tipo de dato en columna fecha_transaccion',          '2023-01-05','Resuelto',  3, 8],
    [5,  'Inconsistencia entre DWH Finanzas y sistema fuente',          '2023-02-18','Resuelto',  1, 2],
    [6,  'Falla en carga incremental de inventario',                    '2023-04-30','Resuelto',  3, 9],
    [7,  'Registros con NPS fuera de rango (-100 a 100)',               '2023-06-12','Resuelto',  6,24],
    [8,  'Stock negativo en reporte de seguridad',                      '2023-07-25','No Resuelto',3, 9],
    [9,  'Discrepancia en tasa de retención entre sistemas',            '2023-08-14','No Resuelto',5, 5],
    [10, 'Timeout en query de tablero ejecutivo',                       '2023-09-01','No Resuelto',1,36],
    [11, 'Datos de ausentismo no actualizados en DWH',                  '2023-10-03','No Resuelto',   4,10],
    [12, 'Errores de encoding en carga de datos externos',              '2023-10-15','No Resuelto',  14, 6],
    [13, 'Pipeline Kafka con mensajes perdidos',                        '2023-11-02','No Resuelto',   8,57],
    [14, 'CAC duplicado por join incorrecto',                           '2023-11-20','No Resuelto',   2,13],
    [15, 'LTV con cálculo erróneo por segmento',                        '2023-12-01','No Resuelto',   5,14],
    [16, 'Tablero de costos con datos desactualizados',                 '2024-01-10','No Resuelto',   1,36],
    [17, 'Proceso ETL de RRHH fallando los lunes',                      '2024-01-22','No Resuelto', 4,59],
    [18, 'Forecast con outliers no filtrados',                          '2024-02-05','No Resuelto',2,58],
    [19, 'Uptime registrado incorrectamente en fuente',                 '2024-02-20','No Resuelto',   8,44],
    [20, 'Incidentes no reportados en tablero de riesgo',               '2024-03-01','No Resuelto',  15,41]
]

columnas = ['id_problema', 'descripcion', 'fecha_origen', 'estado', 'id_fuente', 'id_activo']

# Instanciamos el DataFrame que representa los datos limpios de la capa Bronze
df_incidentes_limpios = pd.DataFrame(datos_reales, columns=columnas)
df_incidentes_limpios.to_csv('bronze.csv', index=False)
print("Dataset inicial 'bronze.csv' exportado correctamente.")

### Construcción del Pipeline de Calidad
A continuación, modularizamos la validación en una función centralizada. Esta función levanta un contexto efímero en memoria, define el contrato de datos (Expectation Suite) con las 5 reglas de negocio de la organización, evalúa el lote completo y bifurca el flujo de datos aislando los registros defectuosos.

In [ ]:
# Declaración de Great Expectations

def ejecutar_pipeline_calidad(df_entrada, titulo_fase):
    """
    Inicializa un contexto efímero en memoria, define las reglas de gobierno de datos,
    corre la validación sobre el DataFrame provisto y expone los resultados detallados.
    """
    print(f" EJECUTANDO BARRERA DE CALIDAD: {titulo_fase}")

    # A. Inicialización del Data Context Efímero
    context = gx.get_context(mode="ephemeral")

    # B. Vinculación física del DataFrame (Data Source -> Asset -> Batch)
    data_source = context.data_sources.add_pandas(name="datasource_incidentes")
    data_asset = data_source.add_dataframe_asset(name="incidentes_asset")
    batch_definition = data_asset.add_batch_definition_whole_dataframe("batch_incidentes")

    # C. Construcción de la Suite de Expectativas lógicas del negocio
    suite = context.suites.add(gx.ExpectationSuite(name="suite_governance_silver"))

    # Regla 1: Integridad de Entidades - Unicidad de la Clave Primaria
    suite.add_expectation(gx.expectations.ExpectColumnValuesToBeUnique(column="id_problema"))

    # Regla 2: Restricción de Dominio - Estados válidos para el ciclo de vida del problema
    suite.add_expectation(gx.expectations.ExpectColumnValuesToBeInSet(
        column="estado", value_set=["Resuelto", "No Resuelto"]
    ))

    # Regla 3: Completitud de Atributos - Obligatoriedad de fecha de origen para auditorías
    suite.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column="fecha_origen"))

    # Regla 4: Consistencia de Formato - Validación de cadena ISO estándar via Expresiones Regulares
    suite.add_expectation(gx.expectations.ExpectColumnValuesToMatchRegex(
        column="fecha_origen", regex=r"^\d{4}-\d{2}-\d{2}$"
    ))

    # Regla 5: Restricción de Rango Referencial - El ID de la fuente física debe ser un entero positivo
    suite.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(
        column="id_fuente", min_value=1
    ))

    # D. Definición de la Validación y ejecución estructurada
    validation_def = context.validation_definitions.add(
        gx.ValidationDefinition(name="validacion_silver", data=batch_definition, suite=suite)
    )

    # Invocación del Runner configurando formato COMPLETE para auditoría fina de anomalías
    results = validation_def.run(
        batch_parameters={"dataframe": df_entrada},
        result_format={
            "result_format": "COMPLETE",
            "unexpected_index_column_names": ["id_problema"],  # Permite mapear qué PK falló exactamente
        }
    )

    # E. Metadata descriptiva para el reporte de salida consolidado
    metadatos_reglas = [
        {"nombre": "Regla 1 (ID Único)", "descripcion": "Valores duplicados en 'id_problema'. PK violada."},
        {"nombre": "Regla 2 (Estado válido)", "descripcion": "Valores fuera del dominio permitido ['Resuelto', 'No Resuelto']."},
        {"nombre": "Regla 3 (Fecha no nula)", "descripcion": "Atributo obligatorio ausente en campo 'fecha_origen'."},
        {"nombre": "Regla 4 (Formato Fecha ISO)", "descripcion": "Formato sintáctico inválido (Requerido: YYYY-MM-DD)."},
        {"nombre": "Regla 5 (ID Fuente > 0)", "descripcion": "Clave foránea referencial inconsistente (Menor a 1)."}
    ]

    print(f"{'REGLA DE ASENCIÓN DE CALIDAD':<35} {'ESTADO':<12} DIAGNÓSTICO TÉCNICO")
    print("─" * 105)

    ids_para_cuarentena = set()

    # F. Iteración estructurada de resultados sobre el grafo de expectativas
    for regla, res in zip(metadatos_reglas, results.results):
        estado_str = "Éxito" if res.success else "Fallo"

        if res.success:
            print(f"{regla['nombre']:<35} {estado_str:<12} Operación limpia sobre la columna.")
        else:
            unexpected_count = res.result.get("unexpected_count", 0)
            unexpected_pct = res.result.get("unexpected_percent", 0.0)
            unexpected_vals = res.result.get("unexpected_list", [])
            unexpected_index = res.result.get("unexpected_index_list", [])

            # Compresión de lista para extraer los IDs de negocio problemáticos
            ids_fallidos = [r["id_problema"] for r in unexpected_index] if unexpected_index else []
            ids_para_cuarentena.update(ids_fallidos)

            print(f"{regla['nombre']:<35} {estado_str:<12} {regla['descripcion']}")
            print(f"  Métricas de Anomalía : {unexpected_count} registros detectados ({unexpected_pct:.1f}%)")
            print(f"  Valores Corruptos    : {unexpected_vals}")
            print(f"  Identificadores (PK) : {ids_fallidos}")
        print()

    # G. Patrón de Ruteo Arquitectural: Bifurcación entre Capa Silver y Quarantine (DLQ)
    print("─" * 105)
    df_dlq = df_entrada[df_entrada["id_problema"].isin(ids_para_cuarentena)]
    df_silver = df_entrada[~df_entrada["id_problema"].isin(ids_para_cuarentena)]

    print(f"\n Metricas finales del pipeline:")
    print(f"  » Total registros procesados en Capa Bronze: {len(df_entrada)}")
    print(f"  » Registros PROMOVIDOS a Capa Silver       : {len(df_silver)}")
    print(f"  » Registros RECHAZADOS (Ruteados a DLQ)    : {len(df_dlq)}")

    if not df_dlq.empty:
        print(f"\n CONTENIDO EN COLA DE LETRAS MUERTAS (DEAD LETTER QUEUE):")
        print(df_dlq.to_string(index=False))

    return df_silver, df_dlq

### Fase 1: Escenario de Control (Datos Limpios)
En primera instancia, el motor evalúa el recorte original de 20 registros extraídos directamente del DML relacional (PostgreSQL). El objetivo es establecer un escenario base de comportamiento.

In [ ]:
# Ejecución Inicial: El dataset viaja puro directamente desde los orígenes validados
df_silver_v1, df_dlq_v1 = ejecutar_pipeline_calidad(df_incidentes_limpios, "Fase PoC Inicial - Dataset Relacional")

**Análisis:** La totalidad del lote fue promovida a la capa Silver sin rechazos (0 registros en DLQ). Este resultado es coherente ya que estos datos provienen de nuestro motor relacional de origen (PostgreSQL) que, por diseño estructural, ya forzaba el cumplimiento de restricciones ACID (claves primarias, foráneas y tipado estricto) antes de su exportación al Data Lake.

### Fase 2: Prueba de Estrés e Inyección de Anomalías
En un entorno real distribuido (Big Data), las fuentes externas pueden romper los contratos de datos por desactualizaciones de software o fallas de tipado en origen. Para simular este escenario, inyectamos intencionalmente 5 anomalías críticas de negocio para evaluar la robustez y capacidad de ruteo de nuestra barrera de calidad.

In [ ]:
# Inyección controlada de registros huérfanos y malformados
datos_sucios = [
    [1,  'Falla de conexión (ID duplicado)',       '2024-01-01', 'No Resuelto',  4, 10], # Falla R1: Clave primaria duplicada
    [21, 'Error desconocido en origen',            '2024-02-15', 'En Análisis', 2, 12], # Falla R2: Estado fuera de dominio
    [22, 'Tabla borrada accidentalmente',          None,         'No Resuelto',  6, 24], # Falla R3: Inyección de nulo prohibido
    [23, 'Lentitud extrema en réplica',            '15/03/2024', 'No Resuelto',  8, 44], # Falla R4: Ruptura de formato ISO
    [24, 'Logs corruptos de auditoría de discos',  '2024-04-10', 'No Resuelto', -5, 15]  # Falla R5: ID referencial fuera de rango
]

df_incidentes_sucios = pd.DataFrame(datos_reales + datos_sucios, columns=columnas)

# Se procesa el consolidado y se evalúa el comportamiento de ruteo
df_silver_v2, df_dlq_v2 = ejecutar_pipeline_calidad(df_incidentes_sucios, "Fase Estresada - Ingesta con Datos Corruptos")

**Análisis:** La prueba de estrés demostró la robustez del Data Quality Gate. El sistema capturó con precisión quirúrgica todas las violaciones.
>
> Resulta especialmente destacable el comportamiento de la herramienta ante la violación de unicidad (Regla 1): al detectar que el registro inyectado compartía el `id_problema = 1` con un registro legítimo y sano, el motor envió ambos a la cuarentena. Esto explica por qué hay 6 registros rechazados frente a 5 inyectados. Esta lógica estricta garantiza que no se procesen datos ambiguos en etapas posteriores, preservando la trazabilidad exacta para que el equipo de soporte pueda auditar el origen de la duplicación.